In [ ]:
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from scipy.stats import chi2, f
from panel_utils import ModelResultsAggregator, run_panel_regressions, run_spec_tests


In [ ]:
###############################
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
###############################
df_reg_analys = pd.read_excel('reg_analys.xlsx')
df_fed_analys = pd.read_excel('fed_analys.xlsx')


# Убираем служебные столбцы индекса
df_reg_analys = df_reg_analys.loc[:, ~df_reg_analys.columns.str.startswith('Unnamed')]
df_fed_analys = df_fed_analys.loc[:, ~df_fed_analys.columns.str.startswith('Unnamed')]

# Приводим даты к datetime
df_reg_analys['Date'] = pd.to_datetime(df_reg_analys['Date'])
df_fed_analys['Date'] = pd.to_datetime(df_fed_analys['Date'])

# Разделим Mon_Shock на позитивный и негативный
df_reg_analys['Mon_Shock_neg'] = df_reg_analys['Mon_Shock'].where(df_reg_analys['Mon_Shock'] < 0, 0)
df_reg_analys['Mon_Shock_pos'] = df_reg_analys['Mon_Shock'].where(df_reg_analys['Mon_Shock'] > 0, 0)

# Объединяем региональные и федеральные данные
fed_extra_cols = [
    col for col in df_fed_analys.columns
    if col not in df_reg_analys.columns and col != 'Region'
]
df_reg = df_reg_analys.merge(
    df_fed_analys[['Date'] + fed_extra_cols],
    on='Date',
    how='left'
)


# Взаимодействия (если требуется)
if 'Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['Mon_Shock_Cl1'] = df_reg['Mon_Shock'] * df_reg['Cluster_1']
if 'Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['Mon_Shock_Cl2'] = df_reg['Mon_Shock'] * df_reg['Cluster_2']
if 'Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['Mon_Shock_Covid'] = df_reg['Mon_Shock'] * df_reg['Covid_dum']

# Кластерные выборки
if 'Cluster_1' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg_clus_one = df_reg[df_reg['Cluster_1'] == 1].copy()
    df_reg_clus_two = df_reg[df_reg['Cluster_2'] == 1].copy()
    df_reg_clus_three = df_reg[(df_reg['Cluster_1'] == 0) & (df_reg['Cluster_2'] == 0)].copy()


In [68]:
###############################
# Построение линейных моделей на панельных данных - общая выборка
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)")
print("Зависимая переменная: d_Int_Rate_FL_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
    'Mon_Shock',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj',
    'Covid_dum',
    'Sank_dum'
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_reg для работы
df_clean = df_reg.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='clustered', cluster_entity=True
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: d_Int_Rate_FL_adj

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_Int_Rate_FL_adj   R-squared:                        0.0839
Estimator:                  PooledOLS   R-squared (Between):              0.7898
No. Observations:                2812   R-squared (Within):               0.0789
Date:                 Ср, дек 24 2025   R-squared (Overall):              0.0839
Time:                        17:50:06   Log-likelihood                   -3976.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      21.383
Entities:                          37   P-value                           0.0000
Avg Obs:                       76.000   Distribution:                 F(12,2800)
Min Obs:                       76.000                                       

In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_1 = pooled_res if pooled_success else None
fe_res_1 = fe_res if fe_success else None
re_res_1 = re_res if re_success else None


In [69]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
from scipy.stats import f

print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -455.5809
p-значение: 1.000000
df: 12

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 1420.0830
p-значение: 0.000000
N (регионов): 37, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -46.1334
p-значение: 1.000000
df: (36, 2763)

Вывод: p >= 0.05 - Pooled адекватна


In [70]:
# Добавляем результаты в сводную таблицу
aggregator_Int_FL = ModelResultsAggregator()

aggregator_Int_FL.add_model_results(
    re_res,
    dependent_variable='d_Int_Rate_FL_adj',
    subsample_name='Общая выборка',
    model_type='RE',
    specification_name='Модель_1',
    se_type = 'Clustered'
)


In [71]:
###############################
# Построение линейных моделей на панельных данных - общая выборка
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)")
print("Зависимая переменная: d_Int_Rate_FL_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
    'Mon_Shock',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj',
    'Covid_dum',
    'Sank_dum',
    'Mon_Shock_Covid'
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_reg для работы
df_clean = df_reg.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='clustered', cluster_entity=True
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: d_Int_Rate_FL_adj

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_Int_Rate_FL_adj   R-squared:                        0.0902
Estimator:                  PooledOLS   R-squared (Between):              0.7926
No. Observations:                2812   R-squared (Within):               0.0852
Date:                 Ср, дек 24 2025   R-squared (Overall):              0.0902
Time:                        17:50:06   Log-likelihood                   -3966.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      21.351
Entities:                          37   P-value                           0.0000
Avg Obs:                       76.000   Distribution:                 F(13,2799)
Min Obs:                       76.000                                       

In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_2 = pooled_res if pooled_success else None
fe_res_2 = fe_res if fe_success else None
re_res_2 = re_res if re_success else None


In [72]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
from scipy.stats import f

print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -343.4245
p-значение: 1.000000
df: 13

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 1420.1131
p-значение: 0.000000
N (регионов): 37, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -46.8270
p-значение: 1.000000
df: (36, 2762)

Вывод: p >= 0.05 - Pooled адекватна


In [73]:
# Добавляем результаты в сводную таблицу

aggregator_Int_FL.add_model_results(
    re_res,
    dependent_variable='d_Int_Rate_FL_adj',
    subsample_name='Общая выборка',
    model_type='RE',
    specification_name='Модель_2',
    se_type = 'Clustered'
)


In [74]:
###############################
# Построение линейных моделей на панельных данных - общая выборка
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)")
print("Зависимая переменная: d_Int_Rate_FL_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
    'Mon_Shock',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj',
    'Covid_dum',
    'Sank_dum',
    'Mon_Shock_Cl1',
    'Mon_Shock_Cl2'
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_reg для работы
df_clean = df_reg.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='clustered', cluster_entity=True
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: d_Int_Rate_FL_adj

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_Int_Rate_FL_adj   R-squared:                        0.0840
Estimator:                  PooledOLS   R-squared (Between):              0.7882
No. Observations:                2812   R-squared (Within):               0.0790
Date:                 Ср, дек 24 2025   R-squared (Overall):              0.0840
Time:                        17:50:07   Log-likelihood                   -3976.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      18.333
Entities:                          37   P-value                           0.0000
Avg Obs:                       76.000   Distribution:                 F(14,2798)
Min Obs:                       76.000                                       

In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_3 = pooled_res if pooled_success else None
fe_res_3 = fe_res if fe_success else None
re_res_3 = re_res if re_success else None


In [75]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
from scipy.stats import f

print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -470.6733
p-значение: 1.000000
df: 14

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 1420.0486
p-значение: 0.000000
N (регионов): 37, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -46.1269
p-значение: 1.000000
df: (36, 2761)

Вывод: p >= 0.05 - Pooled адекватна


In [76]:
# Добавляем результаты в сводную таблицу
aggregator_Int_FL.add_model_results(
    re_res,
    dependent_variable='d_Int_Rate_FL_adj',
    subsample_name='Общая выборка',
    model_type='RE',
    specification_name='Модель_3',
    se_type = 'Clustered'
)

In [98]:
###############################
# Построение линейных моделей на панельных данных - кластер 1
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1)")
print("Зависимая переменная: d_Int_Rate_FL_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
   # 'd_ln_New_Loans_Progr',
    'Mon_Shock',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_one.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1)
Зависимая переменная: d_Int_Rate_FL_adj

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_Int_Rate_FL_adj   R-squared:                        0.1770
Estimator:                  PooledOLS   R-squared (Between):              0.6385
No. Observations:                2280   R-squared (Within):               0.1744
Date:                 Пн, дек 22 2025   R-squared (Overall):              0.1770
Time:                        17:53:02   Log-likelihood                   -3171.4
Cov. Estimator:                Robust                                           
                                        F-statistic:                      44.370
Entities:                          30   P-value                           0.0000
Avg Obs:                       76.000   Distribution:                 F(11,2269)
Min Obs:                       76.000                                           

In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_4 = pooled_res if pooled_success else None
fe_res_4 = fe_res if fe_success else None
re_res_4 = re_res if re_success else None


In [99]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -2.3367
p-значение: 1.000000
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 1149.4296
p-значение: 0.000000
N (регионов): 30, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -44.6675
p-значение: 1.000000
df: (29, 2239)

Вывод: p >= 0.05 - Pooled адекватна


In [100]:
# Добавляем результаты в сводную таблицу
aggregator.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_FL_adj',
    subsample_name='Кластер_1',
    model_type='RE',
    specification_name='Модель_4'
)

In [109]:
###############################
# Построение линейных моделей на панельных данных - кластер 2
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2)")
print("Зависимая переменная: d_Int_Rate_FL_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
   # 'd_ln_New_Loans_Progr',
    'Mon_Shock',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_two.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2)
Зависимая переменная: d_Int_Rate_FL_adj

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_Int_Rate_FL_adj   R-squared:                        0.1883
Estimator:                  PooledOLS   R-squared (Between):              0.8843
No. Observations:                 228   R-squared (Within):               0.1846
Date:                 Пн, дек 22 2025   R-squared (Overall):              0.1883
Time:                        17:53:05   Log-likelihood                   -312.20
Cov. Estimator:                Robust                                           
                                        F-statistic:                      4.5775
Entities:                           3   P-value                           0.0000
Avg Obs:                       76.000   Distribution:                  F(11,217)
Min Obs:                       76.000                                           

In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_5 = pooled_res if pooled_success else None
fe_res_5 = fe_res if fe_success else None
re_res_5 = re_res if re_success else None


In [110]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 0.5269
p-значение: 0.999998
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 115.3463
p-значение: 0.000000
N (регионов): 3, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -43.5550
p-значение: 1.000000
df: (2, 214)

Вывод: p >= 0.05 - Pooled адекватна


In [111]:
# Добавляем результаты в сводную таблицу
aggregator.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_FL_adj',
    subsample_name='Кластер_2',
    model_type='RE',
    specification_name='Модель_7'
)

In [120]:
###############################
# Построение линейных моделей на панельных данных - кластер 3
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3)")
print("Зависимая переменная: d_Int_Rate_FL_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
   # 'd_ln_New_Loans_Progr',
    'Mon_Shock',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj'                               
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_three.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3)
Зависимая переменная: d_Int_Rate_FL_adj

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_Int_Rate_FL_adj   R-squared:                        0.2569
Estimator:                  PooledOLS   R-squared (Between):              0.9367
No. Observations:                 532   R-squared (Within):               0.2517
Date:                 Пн, дек 22 2025   R-squared (Overall):              0.2569
Time:                        17:55:18   Log-likelihood                   -613.05
Cov. Estimator:                Robust                                           
                                        F-statistic:                      16.376
Entities:                           7   P-value                           0.0000
Avg Obs:                       76.000   Distribution:                  F(11,521)
Min Obs:                       76.000                                           

In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_6 = pooled_res if pooled_success else None
fe_res_6 = fe_res if fe_success else None
re_res_6 = re_res if re_success else None


In [121]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 8.7405
p-значение: 0.645829
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 269.1966
p-значение: 0.000000
N (регионов): 7, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -44.7128
p-значение: 1.000000
df: (6, 514)

Вывод: p >= 0.05 - Pooled адекватна


In [122]:
# Добавляем результаты в сводную таблицу
aggregator.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_FL_adj',
    subsample_name='Кластер_3',
    model_type='RE',
    specification_name='Модель_10'
)

In [65]:
###############################
# Построение линейных моделей на панельных данных - общая выборка ROISFIX
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА ROISFIX)")
print("Зависимая переменная: d_Int_Rate_FL_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
   # 'd_ln_New_Loans_Progr',
    'd_ln_ROISFIX',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_reg для работы
df_clean = df_reg.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='clustered', cluster_entity=True
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА) - ROISFIX
Зависимая переменная: d_Int_Rate_FL_adj
ERROR - d_Int_Rate_FL_adj отсутствует в df_reg_temp

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_Int_Rate_FL_adj   R-squared:                        0.1637
Estimator:                  PooledOLS   R-squared (Between):              0.6696
No. Observations:                3040   R-squared (Within):               0.1607
Date:                 Вт, дек 23 2025   R-squared (Overall):              0.1637
Time:                        11:55:53   Log-likelihood                   -4163.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      53.905
Entities:                          40   P-value                           0.0000
Avg Obs:                       76.000   Distribution:                 F(11,3029)
Min Obs:      

In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_7 = pooled_res if pooled_success else None
fe_res_7 = fe_res if fe_success else None
re_res_7 = re_res if re_success else None


In [66]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
from scipy.stats import f

print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -56.2358
p-значение: 1.000000
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 1533.0860
p-значение: 0.000000
N (регионов): 40, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -50.5996
p-значение: 1.000000
df: (39, 2989)

Вывод: p >= 0.05 - Pooled адекватна


In [67]:
# Добавляем результаты в сводную таблицу
aggregator_roisfix = ModelResultsAggregator()

aggregator_roisfix.add_model_results(
    re_res,
    dependent_variable='d_Int_Rate_FL_adj',
    subsample_name='Общая выборка',
    model_type='RE',
    specification_name='Модель_13'
)


In [90]:
###############################
# Построение линейных моделей на панельных данных - кластер 1 ROISFIX
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1 ROISFIX)")
print("Зависимая переменная: d_Int_Rate_FL_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
   # 'd_ln_New_Loans_Progr',
    'd_ln_ROISFIX',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_one.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1 ROISFIX)
Зависимая переменная: d_Int_Rate_FL_adj
ERROR - d_Int_Rate_FL_adj отсутствует в df_reg_temp

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_Int_Rate_FL_adj   R-squared:                        0.1542
Estimator:                  PooledOLS   R-squared (Between):              0.6050
No. Observations:                2280   R-squared (Within):               0.1516
Date:                 Вт, дек 23 2025   R-squared (Overall):              0.1542
Time:                        12:16:38   Log-likelihood                   -3202.7
Cov. Estimator:                Robust                                           
                                        F-statistic:                      37.597
Entities:                          30   P-value                           0.0000
Avg Obs:                       76.000   Distribution:                 F(11,2269)
Min Obs:            

In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_8 = pooled_res if pooled_success else None
fe_res_8 = fe_res if fe_success else None
re_res_8 = re_res if re_success else None


In [91]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 8.1252
p-значение: 0.702044
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 1149.0651
p-значение: 0.000000
N (регионов): 30, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -52.5716
p-значение: 1.000000
df: (29, 2239)

Вывод: p >= 0.05 - Pooled адекватна


In [92]:
# Добавляем результаты в сводную таблицу
aggregator_roisfix.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_FL_adj',
    subsample_name='Кластер_1',
    model_type='RE',
    specification_name='Модель_16'
)

In [103]:
###############################
# Построение линейных моделей на панельных данных - кластер 2 ROISFIX
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2 ROISFIX)")
print("Зависимая переменная: d_Int_Rate_FL_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
   # 'd_ln_New_Loans_Progr',
    'd_ln_ROISFIX',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj'                          
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_two.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2 ROISFIX)
Зависимая переменная: d_Int_Rate_FL_adj
ERROR - d_Int_Rate_FL_adj отсутствует в df_reg_temp

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_Int_Rate_FL_adj   R-squared:                        0.1892
Estimator:                  PooledOLS   R-squared (Between):              0.8509
No. Observations:                 228   R-squared (Within):               0.1857
Date:                 Вт, дек 23 2025   R-squared (Overall):              0.1892
Time:                        12:25:15   Log-likelihood                   -312.08
Cov. Estimator:                Robust                                           
                                        F-statistic:                      4.6036
Entities:                           3   P-value                           0.0000
Avg Obs:                       76.000   Distribution:                  F(11,217)
Min Obs:            

In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_9 = pooled_res if pooled_success else None
fe_res_9 = fe_res if fe_success else None
re_res_9 = re_res if re_success else None


In [104]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 1.3847
p-значение: 0.999743
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 115.2959
p-значение: 0.000000
N (регионов): 3, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -68.6373
p-значение: 1.000000
df: (2, 214)

Вывод: p >= 0.05 - Pooled адекватна


In [105]:
# Добавляем результаты в сводную таблицу
aggregator_roisfix.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_FL_adj',
    subsample_name='Кластер_2',
    model_type='RE',
    specification_name='Модель_19'
)

In [116]:
###############################
# Построение линейных моделей на панельных данных - кластер 3 ROISFIX
# (d_Int_Rate_FL зависимая переменная)
###############################

print("="*70)
print("ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3 ROISFIX)")
print("Зависимая переменная: d_Int_Rate_FL_adj")
print("="*70)

exog_vars_initial = [
    'd_Int_Rate_FL_lag1_adj',
    'd_Cred_nagr_adj',
    'd_D_top5_rozn_adj',
    'd_Fin_Dostup_adj',
    'Credit_impulse_adj',
    'd_Cred_structure_adj',
    'd_Def_Zadolg_Fl_adj',
   # 'd_ln_New_Loans_Progr',
    'd_ln_ROISFIX',
    'd_ln_New_Loans_Fl_adj',
    'd_Bonds_Rate_Correct_5Y_adj',
    'd_exc_rate_adj',
    'Inflation_Expectations_adj'                               
]

dependent_var = 'd_Int_Rate_FL_adj'

# Создаем копию df_reg для работы
df_clean = df_reg_clus_three.copy()  # Ð»Ð¾Ð³Ð¸ÐºÐ° Ð±ÐµÐ· Ð¸Ð·Ð¼ÐµÐ½ÐµÐ½Ð¸Ð¹

# Приводим df_reg к формату с Region и Date как столбцами
if isinstance(df_reg.index, pd.MultiIndex):
    df_reg_temp = df_reg.reset_index()
else:
    df_reg_temp = df_reg.copy()

# Проверяем наличие зависимой переменной

if dependent_var in df_reg_temp.columns:
    
    # Присоединяем зависимую переменную по Region и Date
    df_clean = df_clean.merge(
        df_reg_temp[['Region', 'Date', dependent_var]],
        on=['Region', 'Date'],
        how='left',
        suffixes=('', '_from_reg')
    )
else:
    print(f"ERROR - {dependent_var} отсутствует в df_reg_temp")

# Проверяем независимые переменные
exog_vars_for_regression = []
for var in exog_vars_initial:
    if var in df_clean.columns:
        exog_vars_for_regression.append(var)
    else:
        print(f"  WARNING: {var} исключена")

# Удаляем NaN
cols_to_check = [dependent_var] + exog_vars_for_regression
df_clean = df_clean.dropna(subset=cols_to_check)


# ===== Установка панельного индекса =====
df_clean = df_clean.set_index(['Region', 'Date']).sort_index()

# ===== Подготовка Y и X =====
y = df_clean[[dependent_var]]
X = df_clean[exog_vars_for_regression]


# ===== POOLED OLS =====
pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    y, X, cov_type='robust', cluster_entity=None
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3 ROISFIX)
Зависимая переменная: d_Int_Rate_FL_adj
ERROR - d_Int_Rate_FL_adj отсутствует в df_reg_temp

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:      d_Int_Rate_FL_adj   R-squared:                        0.2384
Estimator:                  PooledOLS   R-squared (Between):              0.9537
No. Observations:                 532   R-squared (Within):               0.2329
Date:                 Вт, дек 23 2025   R-squared (Overall):              0.2384
Time:                        12:32:04   Log-likelihood                   -619.59
Cov. Estimator:                Robust                                           
                                        F-statistic:                      14.827
Entities:                           7   P-value                           0.0000
Avg Obs:                       76.000   Distribution:                  F(11,521)
Min Obs:            

In [ ]:
# Сохраняем результаты модели для экспорта
pooled_res_10 = pooled_res if pooled_success else None
fe_res_10 = fe_res if fe_success else None
re_res_10 = re_res if re_success else None


In [117]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 1.7895
p-значение: 0.999110
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 269.2966
p-значение: 0.000000
N (регионов): 7, T (периодов): 76

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -43.8854
p-значение: 1.000000
df: (6, 514)

Вывод: p >= 0.05 - Pooled адекватна


In [119]:
# Добавляем результаты в сводную таблицу
aggregator_roisfix.add_model_results(
    re_res, 
    dependent_variable='d_Int_Rate_FL_adj',
    subsample_name='Кластер_3',
    model_type='RE',
    specification_name='Модель_22'
)

In [ ]:
from model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

dep_var_name = 'd_Int_Rate_FL_adj'
base_name = dep_var_name
if base_name.startswith('d_'):
    base_name = base_name[2:]
if base_name.endswith('_adj'):
    base_name = base_name[:-4]

results_dir = ensure_results_dir('Results')

model_specs_all = [
    {
        'spec_name': 'Модель_1',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_1,
            'fe': fe_res_1,
            're': re_res_1
        }
    },
    {
        'spec_name': 'Модель_2',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_2,
            'fe': fe_res_2,
            're': re_res_2
        }
    },
    {
        'spec_name': 'Модель_3',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_3,
            'fe': fe_res_3,
            're': re_res_3
        }
    },
    {
        'spec_name': 'Модель_4',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_7,
            'fe': fe_res_7,
            're': re_res_7
        }
    }
]

model_specs_cluster = [
    {
        'spec_name': 'Модель_1',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 1',
        'results': {
            'pooled': pooled_res_4,
            'fe': fe_res_4,
            're': re_res_4
        }
    },
    {
        'spec_name': 'Модель_2',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 1',
        'results': {
            'pooled': pooled_res_8,
            'fe': fe_res_8,
            're': re_res_8
        }
    },
    {
        'spec_name': 'Модель_3',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 2',
        'results': {
            'pooled': pooled_res_5,
            'fe': fe_res_5,
            're': re_res_5
        }
    },
    {
        'spec_name': 'Модель_4',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 2',
        'results': {
            'pooled': pooled_res_9,
            'fe': fe_res_9,
            're': re_res_9
        }
    },
    {
        'spec_name': 'Модель_5',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 3',
        'results': {
            'pooled': pooled_res_6,
            'fe': fe_res_6,
            're': re_res_6
        }
    },
    {
        'spec_name': 'Модель_6',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 3',
        'results': {
            'pooled': pooled_res_10,
            'fe': fe_res_10,
            're': re_res_10
        }
    }
]

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

out_all = os.path.join(results_dir, f"{base_name}_all_data.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)

aggregator_cluster = ModelResultsAggregator()
for spec in model_specs_cluster:
    add_model_set(aggregator_cluster, spec)

out_cluster = os.path.join(results_dir, f"{base_name}_cluster_data.xlsx")
build_and_export(aggregator_cluster, out_cluster, include_pvalues=True, decimals=3)
